# ALS: drug rankings with and without the connectivity-score Monte Carlo p-value filter

Case study `als_NYGC` (NYGC ALS Consortium disease signature). Each connectivity score (CS) in the CS runs of this case study carries a Monte Carlo
p-value (`cs_p_value`). This notebook drops the CS values that are not significant against random gene sets and
compares the resulting drug rankings with the unfiltered ones, in the TSR route (one CS per drug) and in the
per-instance route (one CS per LINCS instance, summarised per compound by sRGES, wsRGES and best).
Sign convention for every score: more negative = stronger predicted reversal of the disease signature.

## Definitions

Terms as computed by the code. Scores are dimensionless ("CS units").

| Term | What it is | Where computed |
|---|---|---|
| **CS** (connectivity score, `bin_chen` method) | Score of one drug signature against the disease signature; more negative = the drug signature reverses the disease signature more strongly. CS units. | `src/connectivity_score.py` |
| **CS p-value** (`cs_p_value`) | Fraction of 1000 random gene sets (same numbers of up- and down-regulated genes, positions drawn without replacement among the reference genes) whose \|CS\| is greater than or equal to the observed \|CS\|. No pseudocount: the value 0 occurs, and the sign of the CS is not used. Resolution 0.001. | `montecarlo_connectivity`, `src/connectivity_score.py` |
| **p'** | (k + 1) / (1000 + 1) with k = `cs_p_value` x 1000; smallest value 0.000999. | `cs_pvalue_filter.flag` |
| **p_adj** | Benjamini-Hochberg adjustment of p' over the rows of one CS run. A CS run is one execution of the CS on one disease signature and one set of drug signatures. | `cs_pvalue_filter.bh` |
| **BH filter** (`bh05`, primary) | Keeps the rows with p_adj <= 0.05. | `cs_pvalue_filter.flag` |
| **Raw filter** (`raw05`, secondary) | Keeps the rows with p' <= 0.05, without multiple-testing adjustment. | `cs_pvalue_filter.flag` |
| **p_adj of the rows at p = 0** | Rows with `cs_p_value` = 0 share the smallest p'; in a run of m rows of which n0 are at p = 0, their p_adj is at most (1 / 1001) x m / n0 (upper bound; the BH step-up takes the minimum over later ranks, so the value can be lower). The bound is <= 0.05 when n0 >= m / 50. | `cs_pvalue_filter.run_stats` |
| **Frozen-offset variant** (`bh05_frozen`) | BH filter, with the surviving instances scored with the dose/time offset table of the unfiltered instances instead of one rebuilt from the survivors; separates the effect of selecting instances from the effect of a changed correction. | `cs_pvalue_filter.frozen_offsets` |
| **Cell-line weight mass lost** | Instance level: 1 - (sum of the weights of the kept used instances) / (sum of the weights of all used instances), over used instances on weighted lines; line level: the same ratio over the weighted lines that keep at least one used instance. Used instance = dose > 0 at 6 or 24 h. | `cs_pvalue_filter.line_stats` |
| **Instance** | One LINCS L1000 level-5 profile: one compound, one cell line, one dose (uM), one time (6 or 24 h). | step `srges` |
| **Configuration** | Which signatures enter the CS: `DEG` (gene-level log2 fold changes), `DEG_LM` (the same restricted to the 978 landmark genes; TSR route only), `mith` (MITHrIL gene perturbations, about 15,800 features), `mith_LM` (MITHrIL perturbations restricted to the landmark genes), `mith_pw` (MITHrIL pathway perturbations). | `per_drug_scores.CONFIGS`, `known_drugs.TSR_CONFIGS` |
| **TSR route**, `<config>_TSR` | One CS per drug on drug signatures collapsed before the CS (drug input `all`, pooled `6h_24h` rows); the ranking score is the CS. After the filter, the drugs with a non-significant CS leave the ranking. | `known_drugs.tsr_rankings` |
| **Per-instance route** | One CS per instance (71 cell lines x configuration); one score per compound after the CS. After the filter, the non-significant instances are dropped and the scores are recomputed from the surviving instances. | `per_drug_scores.collapse` |
| **sRGES** (`<config>_sRGES`) | Per compound: mean over its instances (dose > 0, 6 or 24 h, all cell lines) of the dose/time-adjusted CS (assignment `corrected`; the offset table is rebuilt from the instances present). CS units. | `sRGES.compute_sRGES` |
| **wsRGES** (`<config>_wsRGES`) | The same mean with each instance multiplied by its cell-line weight divided by the largest weight; instances on lines without a weight are left out. CS units. | `sRGES.compute_sRGES` |
| **best** (`<config>_best`) | Per compound: the most negative CS among its instances (dose > 0, 6 or 24 h). CS units. | `per_drug_scores.collapse` |
| **topMed** (`_sRGES_topMed`, `_wsRGES_topMed`, `_sRGES_topQ3`, `_wsRGES_topQ3`) | sRGES / wsRGES restricted to the cell lines above the median (or third-quartile) cutoff of `top_cell_lines`. | `per_drug_scores.collapse` |
| **Ranking** | `<config>_<score>`: compounds sorted by ascending score. Names are normalised to lower case, alphanumerics only (`drug_key`). | `known_drugs` |
| **Percentile** | `rank(pct=True)` x 100 of the score within the ranking's own universe, ascending: 0-100, low = stronger predicted reversal. | `known_drugs.add_ranks` |
| **Universe** | The compounds a ranking scores. Filtering shrinks it, so the percentile of a compound changes even when its score does not. | |
| **Shared universe** | The compounds scored by both rankings that are compared. | `cs_pvalue_filter.compare_rankings` |

In [1]:
import sys, os, warnings, logging
from pathlib import Path
warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

SLUG = 'als_NYGC'
ROOT = Path(os.getcwd()).resolve()
while not (ROOT / 'connectivity_score' / 'input').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import conf
import lfs_style
import case_study_pipeline as cp
from case_study import cs_pvalue_filter as pf
from case_study import known_drugs as kd
from case_study.common import config_colour

lfs_style.apply()
cc = cp.load_case_conf(SLUG)
OUT, IMG = cc.P.oout_dir, cc.P.img_dir
IMG.mkdir(parents=True, exist_ok=True)
REL = '../imgs/case_studies/' + SLUG      # image path relative to this notebook
TOL = 1e-12
assert conf.CS_METHOD == 'bin_chen' and conf.srges_diff_mode == 'corrected'

res = pf.compute(cc)
tsr_names = [n for n in res['rk']['none'] if n.endswith('_TSR')]
pi_names = [n for n in res['rk']['none'] if not n.endswith('_TSR')]
display(Markdown(f"Computed from the CS runs of `{SLUG}`: {len(tsr_names)} TSR rankings and {len(pi_names)} per-instance rankings "
                 f"(srges_diff_mode = `{conf.srges_diff_mode}`, N = {pf.N_ITER} random gene sets per CS, alpha = {pf.ALPHA})."))

Computed from the CS runs of `als_NYGC`: 5 TSR rankings and 28 per-instance rankings (srges_diff_mode = `corrected`, N = 1000 random gene sets per CS, alpha = 0.05).

## 1. Filter

For every CS run the p-value `cs_p_value` is converted to p' = (k + 1) / 1001 (k = `cs_p_value` x 1000, so that p' > 0),
adjusted by Benjamini-Hochberg over all rows of that run, and a row is kept when p_adj <= 0.05 (primary filter, `bh05`);
the secondary filter keeps p' <= 0.05 (`raw05`). The filter uses only the p-value, not the sign of the CS, so
significantly positive CS (drug signatures that resemble the disease signature) are kept as well.

- **TSR route.** Input: the `6h_24h` rows of the drug-input-`all` CS run of each configuration. Operation: BH over those
  rows; drugs that do not pass are dropped. Output: a ranking of the surviving drugs, scores unchanged.
- **Per-instance route.** Input: the CS run of each (configuration, cell line) pair listed in
  `als_NYGC_srges_cs_runs_used.tsv` (the BH adjustment is over the instances of that run). Operation: the instances that
  do not pass are dropped; `per_drug_scores.collapse` (the step-`srges` function) recomputes sRGES, wsRGES, best and
  the top-line variants from the surviving instances, with the dose/time offset table rebuilt from them. A compound with
  no surviving instance leaves the ranking; a collapse variant left with no instance at all gives an empty ranking.
  Output: rankings of the surviving compounds. A second variant (`bh05_frozen`) scores the surviving instances with the
  dose/time offset table of the unfiltered instances; its rankings are compared with those of the rebuilt-offset variant.
- **Drug keys.** Rows whose names normalise to the same `drug_key` are filtered individually and the surviving rows averaged.

## 2. p-value distribution and resolution per configuration

In [2]:
rs = res['run_stats'].copy()
agg = (rs.groupby(['route', 'config'], sort=False)
         .agg(n_runs=('cs_run_id', 'size'), n_rows=('n_rows', 'sum'), n_p0=('n_p0', 'sum'),
              n_keep_bh05=('n_keep_bh05', 'sum'), n_keep_raw05=('n_keep_raw05', 'sum'),
              median_p_adj_at_p0=('p_adj_at_p0', 'median'), median_p_adj_p0_upper_bound=('p_adj_p0_upper_bound', 'median'),
              n_runs_no_row_kept_bh05=('n_keep_bh05', lambda s: int((s == 0).sum())))
         .reset_index())
agg['frac_p0'] = agg['n_p0'] / agg['n_rows']
agg['frac_kept_bh05'] = agg['n_keep_bh05'] / agg['n_rows']
agg['frac_kept_raw05'] = agg['n_keep_raw05'] / agg['n_rows']
agg = agg[['route', 'config', 'n_runs', 'n_rows', 'frac_p0', 'median_p_adj_at_p0', 'median_p_adj_p0_upper_bound',
           'n_runs_no_row_kept_bh05', 'frac_kept_bh05', 'frac_kept_raw05']]
agg.to_csv(OUT / f'{SLUG}_cs_pvalue_filter_resolution.tsv', sep='\t', index=False)
rs.to_csv(OUT / f'{SLUG}_cs_pvalue_filter_runs.tsv', sep='\t', index=False)
display(Markdown(
    f"**Table 1.** Monte Carlo p-values of the CS runs, per route and configuration (`{SLUG}_cs_pvalue_filter_resolution.tsv`; "
    f"one row per route x configuration; per-run values in `{SLUG}_cs_pvalue_filter_runs.tsv`). "
    "`n_runs`: CS runs (per-instance: one per cell line; TSR: one run); `n_rows`: CS values (instances, or drugs at `6h_24h`); "
    "`frac_p0`: fraction of rows with `cs_p_value` = 0 (fraction 0-1); `median_p_adj_at_p0`: median over runs of the p_adj of the rows at p = 0, and `median_p_adj_p0_upper_bound` of its upper bound (1/1001) x m / n0 "
    "(unitless, 0-1; empty when a run has no row at p = 0); `n_runs_no_row_kept_bh05`: runs in which the BH filter keeps "
    "no row; `frac_kept_bh05`, `frac_kept_raw05`: fraction of rows kept (0-1).\n\n" + pf.md_table(agg, 3)))

**Table 1.** Monte Carlo p-values of the CS runs, per route and configuration (`als_NYGC_cs_pvalue_filter_resolution.tsv`; one row per route x configuration; per-run values in `als_NYGC_cs_pvalue_filter_runs.tsv`). `n_runs`: CS runs (per-instance: one per cell line; TSR: one run); `n_rows`: CS values (instances, or drugs at `6h_24h`); `frac_p0`: fraction of rows with `cs_p_value` = 0 (fraction 0-1); `median_p_adj_at_p0`: median over runs of the p_adj of the rows at p = 0, and `median_p_adj_p0_upper_bound` of its upper bound (1/1001) x m / n0 (unitless, 0-1; empty when a run has no row at p = 0); `n_runs_no_row_kept_bh05`: runs in which the BH filter keeps no row; `frac_kept_bh05`, `frac_kept_raw05`: fraction of rows kept (0-1).

| route | config | n_runs | n_rows | frac_p0 | median_p_adj_at_p0 | median_p_adj_p0_upper_bound | n_runs_no_row_kept_bh05 | frac_kept_bh05 | frac_kept_raw05 |
|---|---|---|---|---|---|---|---|---|---|
| tsr | DEG | 1 | 3222 | 0.823 | 0.001 | 0.001 | 0 | 0.858 | 0.858 |
| tsr | DEG_LM | 1 | 3222 | 0.086 | 0.012 | 0.012 | 0 | 0.187 | 0.333 |
| tsr | mith | 1 | 3223 | 0.999 | 1.00e-03 | 1.00e-03 | 0 | 0.999 | 0.999 |
| tsr | mith_LM | 1 | 3223 | 0.102 | 0.010 | 0.010 | 0 | 0.267 | 0.408 |
| tsr | mith_pw | 1 | 3223 | 0.616 | 0.002 | 0.002 | 0 | 0.925 | 0.933 |
| per_instance | DEG | 71 | 66511 | 0.140 | 0.007 | 0.007 | 1 | 0.282 | 0.397 |
| per_instance | mith | 71 | 66511 | 0.999 | 9.99e-04 | 9.99e-04 | 0 | 0.999 | 0.999 |
| per_instance | mith_LM | 71 | 66511 | 0.083 | 0.012 | 0.012 | 1 | 0.202 | 0.359 |
| per_instance | mith_pw | 71 | 66511 | 0.684 | 0.001 | 0.001 | 0 | 0.950 | 0.953 |

In [3]:
cfgs_all = ['DEG', 'DEG_LM', 'mith', 'mith_LM', 'mith_pw']
edges = np.r_[-1e-9, 0.0005, np.arange(0.05, 1.0001, 0.05)]
labels = ['0'] + [f'{a:.2f}-{b:.2f}' for a, b in zip(np.arange(0, 1, 0.05), np.arange(0.05, 1.0001, 0.05))]
fig, axes = plt.subplots(2, 5, figsize=(15, 5.6), sharey=True)
for i, route in enumerate(['tsr', 'per_instance']):
    for j, c in enumerate(cfgs_all):
        ax = axes[i, j]
        p = res['pvals'].get((route, c))
        ax.set_title(f"{c} ({'TSR' if route == 'tsr' else 'per-instance'})", fontsize=9)
        if p is None:
            ax.text(.5, .5, 'not run', ha='center', va='center', transform=ax.transAxes); ax.set_xticks([]); ax.set_yticks([]); continue
        cnt = np.histogram(p, bins=edges)[0]
        ax.bar(range(len(cnt)), cnt / len(p), color=config_colour(c), width=.85)
        ax.set_yscale('log'); ax.set_xticks([0, 1, 5, 10, 15, 20]); ax.set_xticklabels(['0', '0-.05', '.2-.25', '.45-.5', '.7-.75', '.95-1'], fontsize=7, rotation=45)
        ax.set_xlabel('cs_p_value bin', fontsize=8)
        if j == 0: ax.set_ylabel('fraction of rows', fontsize=8)
plt.tight_layout()
f = IMG / f'{SLUG}_cs_pvalue_filter_pvalue_distribution.png'
fig.savefig(f, dpi=130); plt.close(fig)
display(Markdown(f"![p-value distribution]({REL}/{f.name})\n\n**Figure 1.** Distribution of `cs_p_value` per configuration; top row TSR drugs (`6h_24h` rows), "
                 "bottom row per-instance CS pooled over the cell lines. Bars: fraction of rows (0-1, logarithmic axis) at p = 0 (first bar) and in "
                 "20 bins of width 0.05 (0 < p <= 0.05, ..., 0.95 < p <= 1)."))

![p-value distribution](../imgs/case_studies/als_NYGC/als_NYGC_cs_pvalue_filter_pvalue_distribution.png)

**Figure 1.** Distribution of `cs_p_value` per configuration; top row TSR drugs (`6h_24h` rows), bottom row per-instance CS pooled over the cell lines. Bars: fraction of rows (0-1, logarithmic axis) at p = 0 (first bar) and in 20 bins of width 0.05 (0 < p <= 0.05, ..., 0.95 < p <= 1).

In [4]:
sat = agg[agg['frac_p0'] >= 0.5]
unsat = agg[agg['frac_p0'] < 0.5]
txt = ["**Reading of Table 1.**"]
txt.append("Runs in which at least half of the rows have `cs_p_value` = 0: " + (", ".join(f"{r.route} {r.config} (fraction at p = 0: {r.frac_p0:.3f}; fraction kept by BH: {r.frac_kept_bh05:.3f})" for r in sat.itertuples()) or "none") + ".")
txt.append("Runs in which fewer than half of the rows have `cs_p_value` = 0: " + (", ".join(f"{r.route} {r.config} (fraction at p = 0: {r.frac_p0:.3f}; kept by BH: {r.frac_kept_bh05:.3f}; by the raw filter: {r.frac_kept_raw05:.3f})" for r in unsat.itertuples()) or "none") + ".")
txt.append("In a run where `cs_p_value` is 0 for every row, p_adj is the same small number (1/1001 = 0.000999) for every row, so both filters keep every row and the filter does not discriminate between rows; in a run where fewer than 2% of the rows are at p = 0 the upper bound (1/1001) x m / n0 exceeds 0.05 (the bound is not the minimum, so rows at p = 0 can still pass through later ranks).")
display(Markdown("\n\n".join(txt)))

**Reading of Table 1.**

Runs in which at least half of the rows have `cs_p_value` = 0: tsr DEG (fraction at p = 0: 0.823; fraction kept by BH: 0.858), tsr mith (fraction at p = 0: 0.999; fraction kept by BH: 0.999), tsr mith_pw (fraction at p = 0: 0.616; fraction kept by BH: 0.925), per_instance mith (fraction at p = 0: 0.999; fraction kept by BH: 0.999), per_instance mith_pw (fraction at p = 0: 0.684; fraction kept by BH: 0.950).

Runs in which fewer than half of the rows have `cs_p_value` = 0: tsr DEG_LM (fraction at p = 0: 0.086; kept by BH: 0.187; by the raw filter: 0.333), tsr mith_LM (fraction at p = 0: 0.102; kept by BH: 0.267; by the raw filter: 0.408), per_instance DEG (fraction at p = 0: 0.140; kept by BH: 0.282; by the raw filter: 0.397), per_instance mith_LM (fraction at p = 0: 0.083; kept by BH: 0.202; by the raw filter: 0.359).

In a run where `cs_p_value` is 0 for every row, p_adj is the same small number (1/1001 = 0.000999) for every row, so both filters keep every row and the filter does not discriminate between rows; in a run where fewer than 2% of the rows are at p = 0 the upper bound (1/1001) x m / n0 exceeds 0.05 (the bound is not the minimum, so rows at p = 0 can still pass through later ranks).

## 3. Fraction kept, universe size, agreement and top-20 overlap

In [5]:
cmp = {m: pf.compare_rankings(res['rk']['none'], res['rk'][m]) for m in ('bh05', 'raw05', 'bh05f')}
MODE_NAME = {'bh05': 'bh05', 'raw05': 'raw05', 'bh05f': 'bh05_frozen'}
summ = pd.concat([c.assign(filter=MODE_NAME[m]) for m, c in cmp.items()], ignore_index=True)
summ.insert(0, 'config', summ['ranking'].map(lambda n: res['meta'][n][0]))
summ.insert(1, 'route', np.where(summ['ranking'].str.endswith('_TSR'), 'tsr', 'per_instance'))
summ = summ[['filter', 'ranking', 'config', 'route', 'n_before', 'n_after', 'frac_kept', 'n_shared', 'spearman',
             f'top{pf.TOP}_overlap', f'top{pf.TOP}_overlap_shared']]
summ.to_csv(OUT / f'{SLUG}_cs_pvalue_filter_summary.tsv', sep='\t', index=False)
ik = res['inst_kept'].pivot(index='config', columns='mode', values='n_kept')
ik['frac_kept_bh05'] = ik['bh05'] / ik['none']; ik['frac_kept_raw05'] = ik['raw05'] / ik['none']
ik = ik.rename(columns={'none': 'n_instances'}).reset_index()[['config', 'n_instances', 'bh05', 'raw05', 'frac_kept_bh05', 'frac_kept_raw05']]
ik = ik.rename(columns={'bh05': 'n_kept_bh05', 'raw05': 'n_kept_raw05'})
ik.to_csv(OUT / f'{SLUG}_cs_pvalue_filter_instances_kept.tsv', sep='\t', index=False)
display(Markdown(f"**Table 2.** Per-instance route: instances (all dose and time points of the CS runs) kept by each filter, per configuration "
                 f"(`{SLUG}_cs_pvalue_filter_instances_kept.tsv`; fractions 0-1).\n\n" + pf.md_table(ik, 3)))

**Table 2.** Per-instance route: instances (all dose and time points of the CS runs) kept by each filter, per configuration (`als_NYGC_cs_pvalue_filter_instances_kept.tsv`; fractions 0-1).

| config | n_instances | n_kept_bh05 | n_kept_raw05 | frac_kept_bh05 | frac_kept_raw05 |
|---|---|---|---|---|---|
| DEG | 66511 | 18751 | 26414 | 0.282 | 0.397 |
| mith | 66511 | 66450 | 66450 | 0.999 | 0.999 |
| mith_LM | 66511 | 13449 | 23871 | 0.202 | 0.359 |
| mith_pw | 66511 | 63204 | 63416 | 0.950 | 0.953 |

In [6]:
ls = res['line_stats']
tr = []
for n, r in res['tsr_rows'].items():
    for m in ('bh05', 'raw05'):
        kk = r[r[f'keep_{m}']]
        tr.append({'route': 'tsr', 'config': n[:-4], 'filter': m, 'n_rows': len(r), 'n_kept': len(kk), 'n_kept_neg': int((kk['connectivity_score'] < 0).sum()),
                   'n_kept_pos': int((kk['connectivity_score'] > 0).sum())})
sg = pd.concat([pd.DataFrame(tr), ls.assign(route='per_instance', filter=ls['mode'], n_rows=res['inst_kept'].query("mode == 'none'").set_index('config')['n_instances'].reindex(ls['config']).values)
                [['route', 'config', 'filter', 'n_rows', 'n_kept', 'n_kept_neg', 'n_kept_pos']]], ignore_index=True)
sg['frac_kept_neg'] = sg['n_kept_neg'] / sg['n_kept']
sg['frac_kept_pos'] = sg['n_kept_pos'] / sg['n_kept']
sg.to_csv(OUT / f'{SLUG}_cs_pvalue_filter_sign_split.tsv', sep='\t', index=False)
display(Markdown(f"**Table 2b.** Sign of the CS values kept by each filter (`{SLUG}_cs_pvalue_filter_sign_split.tsv`; one row per route, configuration and filter). `n_rows`: CS values of the run(s) "
                 "(TSR: `6h_24h` rows; per-instance: all instances); `n_kept_neg`, `n_kept_pos`: kept values with CS < 0 (reversal) and CS > 0 (count); `frac_kept_neg`, `frac_kept_pos`: their share of the kept values (0-1). "
                 "The p-value is two-sided in |CS|, so the filter keeps significantly positive CS as well; the stored p-value has no one-sided counterpart, "
                 "so the table gives the counts of kept negative CS and no new Monte Carlo was run.\n\n" + pf.md_table(sg, 3)))
cl = ls[ls['mode'] == 'bh05'].copy()
med = {}
for c in cl['config']:
    a_, b_ = res['rk']['none'][f'{c}_sRGES'], res['rk']['bh05'].get(f'{c}_sRGES')
    med[c] = (a_['n_cell_lines'].median(), np.nan if b_ is None or len(b_) == 0 else b_['n_cell_lines'].median())
cl['median_n_cell_lines_per_compound_before'] = cl['config'].map(lambda c: med[c][0])
cl['median_n_cell_lines_per_compound_after'] = cl['config'].map(lambda c: med[c][1])
cl = cl[['config', 'n_lines_used', 'n_lines_kept', 'n_weighted_lines', 'n_weighted_lines_kept', 'frac_line_weight_lost', 'frac_instance_weight_lost',
         'median_n_cell_lines_per_compound_before', 'median_n_cell_lines_per_compound_after']]
cl.to_csv(OUT / f'{SLUG}_cs_pvalue_filter_cell_lines.tsv', sep='\t', index=False)
display(Markdown(f"**Table 2c.** BH filter, per-instance route: cell-line retention per configuration (`{SLUG}_cs_pvalue_filter_cell_lines.tsv`). `n_lines_used`, `n_lines_kept`: cell lines with at least one instance used by sRGES "
                 "(dose > 0 at 6 or 24 h) before and after filtering (count, of 71); `n_weighted_lines`, `n_weighted_lines_kept`: the same for lines with a cell-line weight (the lines wsRGES uses); "
                 "`frac_line_weight_lost`: share (0-1) of the summed weights of the weighted lines that keep no used instance; `frac_instance_weight_lost`: share (0-1) of the weight carried by the used instances on weighted lines "
                 "that is dropped; last two columns: median number of cell lines per compound in `<config>_sRGES` before and after.\n\n" + pf.md_table(cl, 3)))

**Table 2b.** Sign of the CS values kept by each filter (`als_NYGC_cs_pvalue_filter_sign_split.tsv`; one row per route, configuration and filter). `n_rows`: CS values of the run(s) (TSR: `6h_24h` rows; per-instance: all instances); `n_kept_neg`, `n_kept_pos`: kept values with CS < 0 (reversal) and CS > 0 (count); `frac_kept_neg`, `frac_kept_pos`: their share of the kept values (0-1). The p-value is two-sided in |CS|, so the filter keeps significantly positive CS as well; the stored p-value has no one-sided counterpart, so the table gives the counts of kept negative CS and no new Monte Carlo was run.

| route | config | filter | n_rows | n_kept | n_kept_neg | n_kept_pos | frac_kept_neg | frac_kept_pos |
|---|---|---|---|---|---|---|---|---|
| tsr | DEG | bh05 | 3222 | 2763 | 2052 | 711 | 0.743 | 0.257 |
| tsr | DEG | raw05 | 3222 | 2765 | 2052 | 713 | 0.742 | 0.258 |
| tsr | DEG_LM | bh05 | 3222 | 601 | 482 | 119 | 0.802 | 0.198 |
| tsr | DEG_LM | raw05 | 3222 | 1072 | 823 | 249 | 0.768 | 0.232 |
| tsr | mith | bh05 | 3223 | 3221 | 2566 | 655 | 0.797 | 0.203 |
| tsr | mith | raw05 | 3223 | 3221 | 2566 | 655 | 0.797 | 0.203 |
| tsr | mith_LM | bh05 | 3223 | 862 | 811 | 51 | 0.941 | 0.059 |
| tsr | mith_LM | raw05 | 3223 | 1315 | 1226 | 89 | 0.932 | 0.068 |
| tsr | mith_pw | bh05 | 3223 | 2981 | 2146 | 835 | 0.720 | 0.280 |
| tsr | mith_pw | raw05 | 3223 | 3006 | 2162 | 844 | 0.719 | 0.281 |
| per_instance | DEG | bh05 | 66511 | 18751 | 10654 | 8097 | 0.568 | 0.432 |
| per_instance | DEG | raw05 | 66511 | 26414 | 15066 | 11348 | 0.570 | 0.430 |
| per_instance | mith | bh05 | 66511 | 66450 | 40140 | 26310 | 0.604 | 0.396 |
| per_instance | mith | raw05 | 66511 | 66450 | 40140 | 26310 | 0.604 | 0.396 |
| per_instance | mith_LM | bh05 | 66511 | 13449 | 7882 | 5567 | 0.586 | 0.414 |
| per_instance | mith_LM | raw05 | 66511 | 23871 | 14171 | 9700 | 0.594 | 0.406 |
| per_instance | mith_pw | bh05 | 66511 | 63204 | 36750 | 26454 | 0.581 | 0.419 |
| per_instance | mith_pw | raw05 | 66511 | 63416 | 36852 | 26564 | 0.581 | 0.419 |

**Table 2c.** BH filter, per-instance route: cell-line retention per configuration (`als_NYGC_cs_pvalue_filter_cell_lines.tsv`). `n_lines_used`, `n_lines_kept`: cell lines with at least one instance used by sRGES (dose > 0 at 6 or 24 h) before and after filtering (count, of 71); `n_weighted_lines`, `n_weighted_lines_kept`: the same for lines with a cell-line weight (the lines wsRGES uses); `frac_line_weight_lost`: share (0-1) of the summed weights of the weighted lines that keep no used instance; `frac_instance_weight_lost`: share (0-1) of the weight carried by the used instances on weighted lines that is dropped; last two columns: median number of cell lines per compound in `<config>_sRGES` before and after.

| config | n_lines_used | n_lines_kept | n_weighted_lines | n_weighted_lines_kept | frac_line_weight_lost | frac_instance_weight_lost | median_n_cell_lines_per_compound_before | median_n_cell_lines_per_compound_after |
|---|---|---|---|---|---|---|---|---|
| DEG | 70 | 70 | 63 | 63 | 0.000 | 0.721 | 2.000 | 1.000 |
| mith | 70 | 70 | 63 | 63 | 0.000 | 8.32e-04 | 2.000 | 2.000 |
| mith_LM | 70 | 70 | 63 | 63 | 0.000 | 0.800 | 2.000 | 1.000 |
| mith_pw | 70 | 70 | 63 | 63 | 0.000 | 0.049 | 2.000 | 2.000 |

In [7]:
t3 = summ[summ['filter'] == 'bh05'].drop(columns=['filter', 'route'])
display(Markdown(
    f"**Table 3.** BH filter (`bh05`): drug universe before and after filtering and agreement between the unfiltered and the filtered ranking, one row per ranking "
    f"(`{SLUG}_cs_pvalue_filter_summary.tsv`, which also holds the raw filter). `n_before`, `n_after`: compounds in the ranking (count); `frac_kept`: n_after / n_before (0-1); "
    f"`n_shared`: compounds in both; `spearman`: Spearman rho of the scores on the shared compounds (-1 to 1; equal to 1 by construction in the TSR route, where a kept drug keeps its score); "
    f"`top{pf.TOP}_overlap`: compounds common to the {pf.TOP} most negative scores of each ranking (count 0-{pf.TOP}); `top{pf.TOP}_overlap_shared`: the same after restricting both rankings to the shared compounds.\n\n"
    + pf.md_table(t3, 3)))
t3r = summ[summ['filter'] == 'raw05'].drop(columns=['filter', 'route'])
off = summ[summ['route'] == 'per_instance'].pivot(index='ranking', columns='filter', values=['spearman', 'top20_overlap', 'frac_kept'])
off.columns = [f'{a}_{b}' for a, b in off.columns]
off = off.reindex([n for n in pi_names]).reset_index()
rr = {n: pf.compare_rankings({n: res['rk']['bh05'][n]}, {n: res['rk']['bh05f'][n]}).iloc[0] for n in pi_names if n in res['rk']['bh05f']}
off['spearman_rebuilt_vs_frozen'] = off['ranking'].map(lambda n: rr[n]['spearman'] if n in rr else np.nan)
off['abs_diff_spearman_variants'] = (off['spearman_bh05'] - off['spearman_bh05_frozen']).abs()
off = off[['ranking', 'frac_kept_bh05', 'spearman_bh05', 'spearman_bh05_frozen', 'abs_diff_spearman_variants', 'spearman_rebuilt_vs_frozen', 'top20_overlap_bh05', 'top20_overlap_bh05_frozen']]
off.to_csv(OUT / f'{SLUG}_cs_pvalue_filter_offset_variants.tsv', sep='\t', index=False)
chg = off[off['frac_kept_bh05'] < 1]
display(Markdown(
    f"**Table 3b.** Per-instance rankings, BH filter: unfiltered vs filtered Spearman rho (-1 to 1; shared compounds) with the dose/time offset table rebuilt from the surviving instances (`spearman_bh05`) and frozen at the unfiltered one (`spearman_bh05_frozen`); "
    "`abs_diff_spearman_variants`: absolute difference of the two rho; `spearman_rebuilt_vs_frozen`: rho between the two filtered variants; last two columns: top-20 overlap (count 0-20) with the unfiltered ranking "
    f"(`{SLUG}_cs_pvalue_filter_offset_variants.tsv`). Over the rankings the filter changes (`frac_kept_bh05` < 1): largest absolute difference between the two variants' rho {chg['abs_diff_spearman_variants'].max():.3f}; "
    f"lowest `spearman_rebuilt_vs_frozen` {chg['spearman_rebuilt_vs_frozen'].min():.3f}; the `best` rankings do not depend on the offsets.\n\n" + pf.md_table(off, 3)))
display(Markdown("**Table 3c.** Frozen-offset check on the unfiltered instances (`" + f"{SLUG}_cs_pvalue_filter_recomputation_check.tsv`): scoring all instances with the frozen offset table reproduces the unfiltered per-drug scores. `n_compared`: compound x collapse scores (count); `max_abs_diff`: CS units.\n\n" + pf.md_table(res['frozen_check'], 3)))
display(Markdown(
    f"**Table 4.** Raw filter (`raw05`, p' <= 0.05, no multiple-testing adjustment): same columns as Table 3.\n\n" + pf.md_table(t3r, 3)))

**Table 3.** BH filter (`bh05`): drug universe before and after filtering and agreement between the unfiltered and the filtered ranking, one row per ranking (`als_NYGC_cs_pvalue_filter_summary.tsv`, which also holds the raw filter). `n_before`, `n_after`: compounds in the ranking (count); `frac_kept`: n_after / n_before (0-1); `n_shared`: compounds in both; `spearman`: Spearman rho of the scores on the shared compounds (-1 to 1; equal to 1 by construction in the TSR route, where a kept drug keeps its score); `top20_overlap`: compounds common to the 20 most negative scores of each ranking (count 0-20); `top20_overlap_shared`: the same after restricting both rankings to the shared compounds.

| ranking | config | n_before | n_after | frac_kept | n_shared | spearman | top20_overlap | top20_overlap_shared |
|---|---|---|---|---|---|---|---|---|
| DEG_TSR | DEG | 3221 | 2762 | 0.857 | 2762 | 1.000 | 20 | 20 |
| DEG_sRGES | DEG | 12440 | 6893 | 0.554 | 6893 | 0.881 | 4 | 4 |
| DEG_wsRGES | DEG | 12419 | 6731 | 0.542 | 6731 | 0.898 | 3 | 6 |
| DEG_best | DEG | 12440 | 6893 | 0.554 | 6893 | 0.971 | 20 | 20 |
| DEG_sRGES_topQ3 | DEG | 676 | 37 | 0.055 | 37 | 0.969 | 8 | 20 |
| DEG_wsRGES_topQ3 | DEG | 676 | 37 | 0.055 | 37 | 0.969 | 8 | 20 |
| DEG_sRGES_topMed | DEG | 1642 | 489 | 0.298 | 489 | 0.892 | 13 | 13 |
| DEG_wsRGES_topMed | DEG | 1642 | 489 | 0.298 | 489 | 0.894 | 14 | 14 |
| DEG_LM_TSR | DEG_LM | 3221 | 601 | 0.187 | 601 | 1.000 | 20 | 20 |
| mith_TSR | mith | 3221 | 3219 | 0.999 | 3219 | 1.000 | 20 | 20 |
| mith_sRGES | mith | 12440 | 12434 | 1.000 | 12434 | 1.000 | 20 | 20 |
| mith_wsRGES | mith | 12419 | 12413 | 1.000 | 12413 | 1.000 | 20 | 20 |
| mith_best | mith | 12440 | 12434 | 1.000 | 12434 | 1.000 | 20 | 20 |
| mith_sRGES_topQ3 | mith | 676 | 676 | 1.000 | 676 | 1.000 | 20 | 20 |
| mith_wsRGES_topQ3 | mith | 676 | 676 | 1.000 | 676 | 1.000 | 20 | 20 |
| mith_sRGES_topMed | mith | 1642 | 1642 | 1.000 | 1642 | 1.000 | 20 | 20 |
| mith_wsRGES_topMed | mith | 1642 | 1642 | 1.000 | 1642 | 1.000 | 20 | 20 |
| mith_LM_TSR | mith_LM | 3221 | 862 | 0.268 | 862 | 1.000 | 20 | 20 |
| mith_LM_sRGES | mith_LM | 12440 | 5464 | 0.439 | 5464 | 0.826 | 6 | 6 |
| mith_LM_wsRGES | mith_LM | 12419 | 5255 | 0.423 | 5255 | 0.850 | 2 | 2 |
| mith_LM_best | mith_LM | 12440 | 5464 | 0.439 | 5464 | 0.950 | 20 | 20 |
| mith_LM_sRGES_topQ3 | mith_LM | 676 | 40 | 0.059 | 40 | 0.929 | 9 | 19 |
| mith_LM_wsRGES_topQ3 | mith_LM | 676 | 40 | 0.059 | 40 | 0.929 | 9 | 19 |
| mith_LM_sRGES_topMed | mith_LM | 1642 | 501 | 0.305 | 501 | 0.905 | 12 | 12 |
| mith_LM_wsRGES_topMed | mith_LM | 1642 | 501 | 0.305 | 501 | 0.910 | 12 | 14 |
| mith_pw_TSR | mith_pw | 3221 | 2980 | 0.925 | 2980 | 1.000 | 20 | 20 |
| mith_pw_sRGES | mith_pw | 12440 | 12221 | 0.982 | 12221 | 0.995 | 20 | 20 |
| mith_pw_wsRGES | mith_pw | 12419 | 12198 | 0.982 | 12198 | 0.995 | 18 | 18 |
| mith_pw_best | mith_pw | 12440 | 12221 | 0.982 | 12221 | 0.999 | 20 | 20 |
| mith_pw_sRGES_topQ3 | mith_pw | 676 | 648 | 0.959 | 648 | 0.994 | 19 | 19 |
| mith_pw_wsRGES_topQ3 | mith_pw | 676 | 648 | 0.959 | 648 | 0.994 | 19 | 19 |
| mith_pw_sRGES_topMed | mith_pw | 1642 | 1569 | 0.956 | 1569 | 0.992 | 20 | 20 |
| mith_pw_wsRGES_topMed | mith_pw | 1642 | 1569 | 0.956 | 1569 | 0.992 | 18 | 18 |

**Table 3b.** Per-instance rankings, BH filter: unfiltered vs filtered Spearman rho (-1 to 1; shared compounds) with the dose/time offset table rebuilt from the surviving instances (`spearman_bh05`) and frozen at the unfiltered one (`spearman_bh05_frozen`); `abs_diff_spearman_variants`: absolute difference of the two rho; `spearman_rebuilt_vs_frozen`: rho between the two filtered variants; last two columns: top-20 overlap (count 0-20) with the unfiltered ranking (`als_NYGC_cs_pvalue_filter_offset_variants.tsv`). Over the rankings the filter changes (`frac_kept_bh05` < 1): largest absolute difference between the two variants' rho 0.012; lowest `spearman_rebuilt_vs_frozen` 0.956; the `best` rankings do not depend on the offsets.

| ranking | frac_kept_bh05 | spearman_bh05 | spearman_bh05_frozen | abs_diff_spearman_variants | spearman_rebuilt_vs_frozen | top20_overlap_bh05 | top20_overlap_bh05_frozen |
|---|---|---|---|---|---|---|---|
| DEG_sRGES | 0.554 | 0.881 | 0.882 | 0.001 | 0.985 | 4 | 3 |
| DEG_wsRGES | 0.542 | 0.898 | 0.898 | 1.88e-04 | 0.991 | 3 | 3 |
| DEG_best | 0.554 | 0.971 | 0.971 | 0.000 | 1.000 | 20 | 20 |
| DEG_sRGES_topQ3 | 0.055 | 0.969 | 0.971 | 0.002 | 0.959 | 8 | 8 |
| DEG_wsRGES_topQ3 | 0.055 | 0.969 | 0.971 | 0.002 | 0.959 | 8 | 8 |
| DEG_sRGES_topMed | 0.298 | 0.892 | 0.880 | 0.012 | 0.984 | 13 | 11 |
| DEG_wsRGES_topMed | 0.298 | 0.894 | 0.885 | 0.009 | 0.984 | 14 | 13 |
| mith_sRGES | 1.000 | 1.000 | 1.000 | 1.51e-07 | 1.000 | 20 | 20 |
| mith_wsRGES | 1.000 | 1.000 | 1.000 | 4.99e-08 | 1.000 | 20 | 20 |
| mith_best | 1.000 | 1.000 | 1.000 | 0.000 | 1.000 | 20 | 20 |
| mith_sRGES_topQ3 | 1.000 | 1.000 | 1.000 | 7.77e-07 | 1.000 | 20 | 20 |
| mith_wsRGES_topQ3 | 1.000 | 1.000 | 1.000 | 7.77e-07 | 1.000 | 20 | 20 |
| mith_sRGES_topMed | 1.000 | 1.000 | 1.000 | 3.17e-07 | 1.000 | 20 | 20 |
| mith_wsRGES_topMed | 1.000 | 1.000 | 1.000 | 3.28e-07 | 1.000 | 20 | 20 |
| mith_LM_sRGES | 0.439 | 0.826 | 0.831 | 0.005 | 0.969 | 6 | 4 |
| mith_LM_wsRGES | 0.423 | 0.850 | 0.850 | 1.70e-05 | 0.984 | 2 | 2 |
| mith_LM_best | 0.439 | 0.950 | 0.950 | 0.000 | 1.000 | 20 | 20 |
| mith_LM_sRGES_topQ3 | 0.059 | 0.929 | 0.941 | 0.011 | 0.956 | 9 | 9 |
| mith_LM_wsRGES_topQ3 | 0.059 | 0.929 | 0.941 | 0.011 | 0.956 | 9 | 9 |
| mith_LM_sRGES_topMed | 0.305 | 0.905 | 0.905 | 4.68e-04 | 0.984 | 12 | 10 |
| mith_LM_wsRGES_topMed | 0.305 | 0.910 | 0.910 | 1.00e-04 | 0.986 | 12 | 10 |
| mith_pw_sRGES | 0.982 | 0.995 | 0.995 | 6.00e-06 | 1.000 | 20 | 20 |
| mith_pw_wsRGES | 0.982 | 0.995 | 0.995 | 7.04e-06 | 1.000 | 18 | 18 |
| mith_pw_best | 0.982 | 0.999 | 0.999 | 0.000 | 1.000 | 20 | 20 |
| mith_pw_sRGES_topQ3 | 0.959 | 0.994 | 0.994 | 4.19e-05 | 1.000 | 19 | 18 |
| mith_pw_wsRGES_topQ3 | 0.959 | 0.994 | 0.994 | 4.19e-05 | 1.000 | 19 | 18 |
| mith_pw_sRGES_topMed | 0.956 | 0.992 | 0.992 | 3.06e-05 | 1.000 | 20 | 19 |
| mith_pw_wsRGES_topMed | 0.956 | 0.992 | 0.992 | 2.34e-05 | 1.000 | 18 | 18 |

**Table 3c.** Frozen-offset check on the unfiltered instances (`als_NYGC_cs_pvalue_filter_recomputation_check.tsv`): scoring all instances with the frozen offset table reproduces the unfiltered per-drug scores. `n_compared`: compound x collapse scores (count); `max_abs_diff`: CS units.

| config | n_compared | max_abs_diff |
|---|---|---|
| DEG | 41943 | 0.000 |
| mith | 41943 | 0.000 |
| mith_LM | 41943 | 0.000 |
| mith_pw | 41943 | 0.000 |

**Table 4.** Raw filter (`raw05`, p' <= 0.05, no multiple-testing adjustment): same columns as Table 3.

| ranking | config | n_before | n_after | frac_kept | n_shared | spearman | top20_overlap | top20_overlap_shared |
|---|---|---|---|---|---|---|---|---|
| DEG_TSR | DEG | 3221 | 2764 | 0.858 | 2764 | 1.000 | 20 | 20 |
| DEG_sRGES | DEG | 12440 | 8178 | 0.657 | 8178 | 0.906 | 5 | 5 |
| DEG_wsRGES | DEG | 12419 | 8047 | 0.648 | 8047 | 0.917 | 5 | 8 |
| DEG_best | DEG | 12440 | 8178 | 0.657 | 8178 | 0.980 | 20 | 20 |
| DEG_sRGES_topQ3 | DEG | 676 | 169 | 0.250 | 169 | 0.924 | 14 | 14 |
| DEG_wsRGES_topQ3 | DEG | 676 | 169 | 0.250 | 169 | 0.924 | 14 | 14 |
| DEG_sRGES_topMed | DEG | 1642 | 733 | 0.446 | 733 | 0.913 | 14 | 14 |
| DEG_wsRGES_topMed | DEG | 1642 | 733 | 0.446 | 733 | 0.907 | 14 | 14 |
| DEG_LM_TSR | DEG_LM | 3221 | 1072 | 0.333 | 1072 | 1.000 | 20 | 20 |
| mith_TSR | mith | 3221 | 3219 | 0.999 | 3219 | 1.000 | 20 | 20 |
| mith_sRGES | mith | 12440 | 12434 | 1.000 | 12434 | 1.000 | 20 | 20 |
| mith_wsRGES | mith | 12419 | 12413 | 1.000 | 12413 | 1.000 | 20 | 20 |
| mith_best | mith | 12440 | 12434 | 1.000 | 12434 | 1.000 | 20 | 20 |
| mith_sRGES_topQ3 | mith | 676 | 676 | 1.000 | 676 | 1.000 | 20 | 20 |
| mith_wsRGES_topQ3 | mith | 676 | 676 | 1.000 | 676 | 1.000 | 20 | 20 |
| mith_sRGES_topMed | mith | 1642 | 1642 | 1.000 | 1642 | 1.000 | 20 | 20 |
| mith_wsRGES_topMed | mith | 1642 | 1642 | 1.000 | 1642 | 1.000 | 20 | 20 |
| mith_LM_TSR | mith_LM | 3221 | 1315 | 0.408 | 1315 | 1.000 | 20 | 20 |
| mith_LM_sRGES | mith_LM | 12440 | 7676 | 0.617 | 7676 | 0.880 | 9 | 9 |
| mith_LM_wsRGES | mith_LM | 12419 | 7546 | 0.608 | 7546 | 0.895 | 7 | 7 |
| mith_LM_best | mith_LM | 12440 | 7676 | 0.617 | 7676 | 0.971 | 20 | 20 |
| mith_LM_sRGES_topQ3 | mith_LM | 676 | 212 | 0.314 | 212 | 0.969 | 13 | 13 |
| mith_LM_wsRGES_topQ3 | mith_LM | 676 | 212 | 0.314 | 212 | 0.969 | 13 | 13 |
| mith_LM_sRGES_topMed | mith_LM | 1642 | 815 | 0.496 | 815 | 0.930 | 15 | 15 |
| mith_LM_wsRGES_topMed | mith_LM | 1642 | 815 | 0.496 | 815 | 0.930 | 14 | 14 |
| mith_pw_TSR | mith_pw | 3221 | 3005 | 0.933 | 3005 | 1.000 | 20 | 20 |
| mith_pw_sRGES | mith_pw | 12440 | 12231 | 0.983 | 12231 | 0.995 | 20 | 20 |
| mith_pw_wsRGES | mith_pw | 12419 | 12209 | 0.983 | 12209 | 0.996 | 18 | 18 |
| mith_pw_best | mith_pw | 12440 | 12231 | 0.983 | 12231 | 0.999 | 20 | 20 |
| mith_pw_sRGES_topQ3 | mith_pw | 676 | 650 | 0.962 | 650 | 0.994 | 19 | 19 |
| mith_pw_wsRGES_topQ3 | mith_pw | 676 | 650 | 0.962 | 650 | 0.994 | 19 | 19 |
| mith_pw_sRGES_topMed | mith_pw | 1642 | 1573 | 0.958 | 1573 | 0.992 | 19 | 19 |
| mith_pw_wsRGES_topMed | mith_pw | 1642 | 1573 | 0.958 | 1573 | 0.992 | 18 | 18 |

In [8]:
order = list(res['rk']['none'])
x = np.arange(len(order))
fig, ax = plt.subplots(figsize=(14, 4.6))
for off, m, hatch in [(-.2, 'bh05', None), (.2, 'raw05', '//')]:
    d = summ[summ['filter'] == m].set_index('ranking').loc[order]
    ax.bar(x + off, d['frac_kept'], width=.38, color=[config_colour(res['meta'][n][0]) for n in order], hatch=hatch,
           edgecolor='k', linewidth=.4)
ax.set_xticks(x); ax.set_xticklabels(order, rotation=70, ha='right', fontsize=8)
ax.set_ylabel('fraction of compounds kept (0-1)'); ax.set_ylim(0, 1.05)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor='0.8', edgecolor='k', label="BH-adjusted p <= 0.05"), Patch(facecolor='0.8', edgecolor='k', hatch='//', label="raw p' <= 0.05")], fontsize=8)
plt.tight_layout()
f = IMG / f'{SLUG}_cs_pvalue_filter_fraction_kept.png'
fig.savefig(f, dpi=130); plt.close(fig)
display(Markdown(f"![fraction kept]({REL}/{f.name})\n\n**Figure 2.** Fraction of the compounds of each unfiltered ranking that remain after the filter "
                 "(0-1); plain bars BH-adjusted p <= 0.05, hatched bars raw p' <= 0.05; colour = configuration."))

b = summ[summ['filter'] == 'bh05'].set_index('ranking').loc[order]
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
cols = [config_colour(res['meta'][n][0]) for n in order]
bf = summ[summ['filter'] == 'bh05_frozen'].set_index('ranking').loc[order]
axes[0].bar(x - .2, b['spearman'], width=.38, color=cols, edgecolor='k', linewidth=.4); axes[0].bar(x + .2, bf['spearman'], width=.38, color=cols, edgecolor='k', linewidth=.4, hatch='//'); axes[0].set_ylabel('Spearman rho, shared compounds'); axes[0].set_ylim(min(0, np.nanmin(b['spearman']) - .05), 1.02)
axes[1].bar(x - .2, b[f'top{pf.TOP}_overlap'], width=.38, color=cols, edgecolor='k', linewidth=.4); axes[1].bar(x + .2, bf[f'top{pf.TOP}_overlap'], width=.38, color=cols, edgecolor='k', linewidth=.4, hatch='//'); axes[1].set_ylabel(f'top-{pf.TOP} overlap (count)'); axes[1].set_ylim(0, pf.TOP)
axes[1].set_xticks(x); axes[1].set_xticklabels(order, rotation=70, ha='right', fontsize=8)
plt.tight_layout()
f = IMG / f'{SLUG}_cs_pvalue_filter_agreement.png'
fig.savefig(f, dpi=130); plt.close(fig)
display(Markdown(f"![agreement]({REL}/{f.name})\n\n**Figure 3.** BH filter: agreement between the unfiltered and the filtered ranking. Plain bars: dose/time offsets rebuilt from the surviving instances; hatched bars: offsets frozen at the unfiltered table (equal to plain bars for TSR rankings). Top: Spearman rho of the scores on the compounds "
                 f"present in both (-1 to 1). Bottom: number of compounds (0-{pf.TOP}) shared by the {pf.TOP} most negative scores of the two rankings. Colour = configuration."))

![fraction kept](../imgs/case_studies/als_NYGC/als_NYGC_cs_pvalue_filter_fraction_kept.png)

**Figure 2.** Fraction of the compounds of each unfiltered ranking that remain after the filter (0-1); plain bars BH-adjusted p <= 0.05, hatched bars raw p' <= 0.05; colour = configuration.

![agreement](../imgs/case_studies/als_NYGC/als_NYGC_cs_pvalue_filter_agreement.png)

**Figure 3.** BH filter: agreement between the unfiltered and the filtered ranking. Plain bars: dose/time offsets rebuilt from the surviving instances; hatched bars: offsets frozen at the unfiltered table (equal to plain bars for TSR rankings). Top: Spearman rho of the scores on the compounds present in both (-1 to 1). Bottom: number of compounds (0-20) shared by the 20 most negative scores of the two rankings. Colour = configuration.

## 4. Known drugs before and after the filter

In [9]:
known = {d: (c, [kd.key([d])[0]]) for d, c in cc.KNOWN_DRUGS.items()}
DETAIL_CLASSES = None
KNOWN_TEXT = f"{len(known)} known ALS drugs (`KNOWN_DRUGS` of the case-study specification): " + ", ".join(f"{d} ({c})" for d, (c, _) in known.items())
kn = pd.concat([pf.known_table(res['rk']['none'], res['rk'][m], known, m) for m in ('bh05', 'raw05')], ignore_index=True)
kn.to_csv(OUT / f'{SLUG}_cs_pvalue_filter_known_drugs.tsv', sep='\t', index=False)
display(Markdown(f"Known drugs: {KNOWN_TEXT}. Small n: the numbers below describe where these drugs fall and do not measure how well a ranking works."))

Known drugs: 5 known ALS drugs (`KNOWN_DRUGS` of the case-study specification): riluzole (approved, disease-modifying), edaravone (approved, disease-modifying), dextromethorphan (approved, symptomatic (Nuedexta component)), quinidine (approved, symptomatic (Nuedexta component)), phenylbutyrate (failed phase 3 (AMX0035 component)). Small n: the numbers below describe where these drugs fall and do not measure how well a ranking works.

In [10]:
k = kn[(kn['filter'] == 'bh05') & (kn['status'] != 'not scored')]
ag = (k.groupby(['ranking', 'class'], sort=False)
        .apply(lambda d: pd.Series({'n_scored': len(d), 'n_kept': int((d['status'] == 'kept').sum()),
                                    'median_percentile_before_kept': d.loc[d['status'] == 'kept', 'percentile_before'].median(),
                                    'median_percentile_after': d.loc[d['status'] == 'kept', 'percentile_after'].median()}), include_groups=False)
        .reset_index())
ag[['n_scored', 'n_kept']] = ag[['n_scored', 'n_kept']].astype(int)
display(Markdown("**Table 5.** BH filter: known drugs per ranking and class (full table, also for the raw filter: "
                 f"`{SLUG}_cs_pvalue_filter_known_drugs.tsv`). `n_scored`: known drugs scored by the unfiltered ranking (count); `n_kept`: of those, drugs that remain after the filter; "
                 "medians over the kept drugs of the percentile (0-100, low = stronger predicted reversal) in the unfiltered and in the filtered ranking, each within its own universe.\n\n" + pf.md_table(ag, 1)))
d = k if DETAIL_CLASSES is None else k[k['class'].isin(DETAIL_CLASSES)]
dd = d[['ranking', 'drug', 'n_before', 'rank_before', 'percentile_before', 'n_after', 'rank_after', 'percentile_after', 'status']].copy()
for c in ['rank_before', 'rank_after']:
    dd[c] = dd[c].map(lambda v: '' if pd.isna(v) else f'{int(v)}')
display(Markdown("**Table 6.** BH filter: individual known drugs" + ("" if DETAIL_CLASSES is None else " of class " + ", ".join(DETAIL_CLASSES)) +
                 ", one row per ranking. `rank_*`: position in the ranking (1 = most negative score); `n_*`: compounds in the ranking (count); `percentile_*`: 0-100, low = stronger predicted reversal; "
                 "`status`: `kept` / `removed` by the filter.\n\n" + pf.md_table(dd, 1)))

**Table 5.** BH filter: known drugs per ranking and class (full table, also for the raw filter: `als_NYGC_cs_pvalue_filter_known_drugs.tsv`). `n_scored`: known drugs scored by the unfiltered ranking (count); `n_kept`: of those, drugs that remain after the filter; medians over the kept drugs of the percentile (0-100, low = stronger predicted reversal) in the unfiltered and in the filtered ranking, each within its own universe.

| ranking | class | n_scored | n_kept | median_percentile_before_kept | median_percentile_after |
|---|---|---|---|---|---|
| DEG_TSR | approved, disease-modifying | 1 | 1 | 11.1 | 12.9 |
| DEG_TSR | approved, symptomatic (Nuedexta component) | 2 | 0 |  |  |
| DEG_TSR | failed phase 3 (AMX0035 component) | 1 | 1 | 88.3 | 86.4 |
| DEG_sRGES | approved, disease-modifying | 2 | 2 | 39.9 | 47.4 |
| DEG_sRGES | approved, symptomatic (Nuedexta component) | 2 | 2 | 64.5 | 77.4 |
| DEG_sRGES | failed phase 3 (AMX0035 component) | 1 | 0 |  |  |
| DEG_wsRGES | approved, disease-modifying | 2 | 2 | 44.5 | 52.9 |
| DEG_wsRGES | approved, symptomatic (Nuedexta component) | 2 | 2 | 68.5 | 75.0 |
| DEG_wsRGES | failed phase 3 (AMX0035 component) | 1 | 0 |  |  |
| DEG_best | approved, disease-modifying | 2 | 2 | 18.3 | 32.9 |
| DEG_best | approved, symptomatic (Nuedexta component) | 2 | 2 | 42.2 | 70.7 |
| DEG_best | failed phase 3 (AMX0035 component) | 1 | 0 |  |  |
| DEG_sRGES_topQ3 | approved, disease-modifying | 1 | 0 |  |  |
| DEG_sRGES_topQ3 | failed phase 3 (AMX0035 component) | 1 | 0 |  |  |
| DEG_wsRGES_topQ3 | approved, disease-modifying | 1 | 0 |  |  |
| DEG_wsRGES_topQ3 | failed phase 3 (AMX0035 component) | 1 | 0 |  |  |
| DEG_sRGES_topMed | approved, disease-modifying | 1 | 1 | 77.4 | 84.9 |
| DEG_sRGES_topMed | approved, symptomatic (Nuedexta component) | 2 | 1 | 96.2 | 76.9 |
| DEG_sRGES_topMed | failed phase 3 (AMX0035 component) | 1 | 0 |  |  |
| DEG_wsRGES_topMed | approved, disease-modifying | 1 | 1 | 76.0 | 82.0 |
| DEG_wsRGES_topMed | approved, symptomatic (Nuedexta component) | 2 | 1 | 95.2 | 76.5 |
| DEG_wsRGES_topMed | failed phase 3 (AMX0035 component) | 1 | 0 |  |  |
| DEG_LM_TSR | approved, disease-modifying | 1 | 1 | 15.2 | 77.5 |
| DEG_LM_TSR | approved, symptomatic (Nuedexta component) | 2 | 0 |  |  |
| DEG_LM_TSR | failed phase 3 (AMX0035 component) | 1 | 0 |  |  |
| mith_TSR | approved, disease-modifying | 1 | 1 | 13.7 | 13.7 |
| mith_TSR | approved, symptomatic (Nuedexta component) | 2 | 2 | 66.6 | 66.6 |
| mith_TSR | failed phase 3 (AMX0035 component) | 1 | 1 | 85.5 | 85.5 |
| mith_sRGES | approved, disease-modifying | 2 | 2 | 31.6 | 31.7 |
| mith_sRGES | approved, symptomatic (Nuedexta component) | 2 | 2 | 71.9 | 71.9 |
| mith_sRGES | failed phase 3 (AMX0035 component) | 1 | 1 | 80.1 | 80.1 |
| mith_wsRGES | approved, disease-modifying | 2 | 2 | 20.2 | 20.2 |
| mith_wsRGES | approved, symptomatic (Nuedexta component) | 2 | 2 | 78.5 | 78.5 |
| mith_wsRGES | failed phase 3 (AMX0035 component) | 1 | 1 | 94.5 | 94.5 |
| mith_best | approved, disease-modifying | 2 | 2 | 4.4 | 4.4 |
| mith_best | approved, symptomatic (Nuedexta component) | 2 | 2 | 31.4 | 31.4 |
| mith_best | failed phase 3 (AMX0035 component) | 1 | 1 | 24.6 | 24.6 |
| mith_sRGES_topQ3 | approved, disease-modifying | 1 | 1 | 5.8 | 5.6 |
| mith_sRGES_topQ3 | failed phase 3 (AMX0035 component) | 1 | 1 | 86.2 | 86.2 |
| mith_wsRGES_topQ3 | approved, disease-modifying | 1 | 1 | 5.8 | 5.6 |
| mith_wsRGES_topQ3 | failed phase 3 (AMX0035 component) | 1 | 1 | 86.2 | 86.2 |
| mith_sRGES_topMed | approved, disease-modifying | 1 | 1 | 8.8 | 8.9 |
| mith_sRGES_topMed | approved, symptomatic (Nuedexta component) | 2 | 2 | 86.2 | 86.2 |
| mith_sRGES_topMed | failed phase 3 (AMX0035 component) | 1 | 1 | 89.8 | 89.8 |
| mith_wsRGES_topMed | approved, disease-modifying | 1 | 1 | 5.1 | 5.2 |
| mith_wsRGES_topMed | approved, symptomatic (Nuedexta component) | 2 | 2 | 84.2 | 84.2 |
| mith_wsRGES_topMed | failed phase 3 (AMX0035 component) | 1 | 1 | 92.5 | 92.5 |
| mith_LM_TSR | approved, disease-modifying | 1 | 1 | 11.4 | 42.7 |
| mith_LM_TSR | approved, symptomatic (Nuedexta component) | 2 | 0 |  |  |
| mith_LM_TSR | failed phase 3 (AMX0035 component) | 1 | 0 |  |  |
| mith_LM_sRGES | approved, disease-modifying | 2 | 2 | 26.4 | 32.6 |
| mith_LM_sRGES | approved, symptomatic (Nuedexta component) | 2 | 2 | 68.2 | 78.4 |
| mith_LM_sRGES | failed phase 3 (AMX0035 component) | 1 | 1 | 71.2 | 51.9 |
| mith_LM_wsRGES | approved, disease-modifying | 2 | 2 | 18.6 | 27.4 |
| mith_LM_wsRGES | approved, symptomatic (Nuedexta component) | 2 | 2 | 71.6 | 70.5 |
| mith_LM_wsRGES | failed phase 3 (AMX0035 component) | 1 | 1 | 75.4 | 58.0 |
| mith_LM_best | approved, disease-modifying | 2 | 2 | 13.5 | 30.7 |
| mith_LM_best | approved, symptomatic (Nuedexta component) | 2 | 2 | 32.1 | 68.4 |
| mith_LM_best | failed phase 3 (AMX0035 component) | 1 | 1 | 24.0 | 53.9 |
| mith_LM_sRGES_topQ3 | approved, disease-modifying | 1 | 0 |  |  |
| mith_LM_sRGES_topQ3 | failed phase 3 (AMX0035 component) | 1 | 0 |  |  |
| mith_LM_wsRGES_topQ3 | approved, disease-modifying | 1 | 0 |  |  |
| mith_LM_wsRGES_topQ3 | failed phase 3 (AMX0035 component) | 1 | 0 |  |  |
| mith_LM_sRGES_topMed | approved, disease-modifying | 1 | 0 |  |  |
| mith_LM_sRGES_topMed | approved, symptomatic (Nuedexta component) | 2 | 1 | 97.1 | 87.6 |
| mith_LM_sRGES_topMed | failed phase 3 (AMX0035 component) | 1 | 1 | 69.1 | 72.9 |
| mith_LM_wsRGES_topMed | approved, disease-modifying | 1 | 0 |  |  |
| mith_LM_wsRGES_topMed | approved, symptomatic (Nuedexta component) | 2 | 1 | 95.6 | 86.2 |
| mith_LM_wsRGES_topMed | failed phase 3 (AMX0035 component) | 1 | 1 | 68.6 | 72.9 |
| mith_pw_TSR | approved, disease-modifying | 1 | 1 | 12.8 | 13.8 |
| mith_pw_TSR | approved, symptomatic (Nuedexta component) | 2 | 2 | 43.1 | 46.6 |
| mith_pw_TSR | failed phase 3 (AMX0035 component) | 1 | 1 | 97.0 | 96.7 |
| mith_pw_sRGES | approved, disease-modifying | 2 | 2 | 39.5 | 40.5 |
| mith_pw_sRGES | approved, symptomatic (Nuedexta component) | 2 | 2 | 66.2 | 66.4 |
| mith_pw_sRGES | failed phase 3 (AMX0035 component) | 1 | 1 | 68.1 | 67.6 |
| mith_pw_wsRGES | approved, disease-modifying | 2 | 2 | 27.8 | 28.4 |
| mith_pw_wsRGES | approved, symptomatic (Nuedexta component) | 2 | 2 | 67.0 | 67.7 |
| mith_pw_wsRGES | failed phase 3 (AMX0035 component) | 1 | 1 | 94.3 | 94.3 |
| mith_pw_best | approved, disease-modifying | 2 | 2 | 26.9 | 27.4 |
| mith_pw_best | approved, symptomatic (Nuedexta component) | 2 | 2 | 29.5 | 30.0 |
| mith_pw_best | failed phase 3 (AMX0035 component) | 1 | 1 | 25.5 | 26.0 |
| mith_pw_sRGES_topQ3 | approved, disease-modifying | 1 | 1 | 3.3 | 3.7 |
| mith_pw_sRGES_topQ3 | failed phase 3 (AMX0035 component) | 1 | 1 | 74.9 | 74.2 |
| mith_pw_wsRGES_topQ3 | approved, disease-modifying | 1 | 1 | 3.3 | 3.7 |
| mith_pw_wsRGES_topQ3 | failed phase 3 (AMX0035 component) | 1 | 1 | 74.9 | 74.2 |
| mith_pw_sRGES_topMed | approved, disease-modifying | 1 | 1 | 3.8 | 4.3 |
| mith_pw_sRGES_topMed | approved, symptomatic (Nuedexta component) | 2 | 0 |  |  |
| mith_pw_sRGES_topMed | failed phase 3 (AMX0035 component) | 1 | 1 | 86.8 | 86.6 |
| mith_pw_wsRGES_topMed | approved, disease-modifying | 1 | 1 | 2.6 | 3.1 |
| mith_pw_wsRGES_topMed | approved, symptomatic (Nuedexta component) | 2 | 0 |  |  |
| mith_pw_wsRGES_topMed | failed phase 3 (AMX0035 component) | 1 | 1 | 89.3 | 88.8 |

**Table 6.** BH filter: individual known drugs, one row per ranking. `rank_*`: position in the ranking (1 = most negative score); `n_*`: compounds in the ranking (count); `percentile_*`: 0-100, low = stronger predicted reversal; `status`: `kept` / `removed` by the filter.

| ranking | drug | n_before | rank_before | percentile_before | n_after | rank_after | percentile_after | status |
|---|---|---|---|---|---|---|---|---|
| DEG_TSR | riluzole | 3221 | 356 | 11.1 | 2762 | 356 | 12.9 | kept |
| DEG_TSR | dextromethorphan | 3221 | 2425 | 75.3 | 2762 |  |  | removed |
| DEG_TSR | quinidine | 3221 | 2301 | 71.4 | 2762 |  |  | removed |
| DEG_TSR | phenylbutyrate | 3221 | 2845 | 88.3 | 2762 | 2386 | 86.4 | kept |
| DEG_sRGES | riluzole | 12440 | 7999 | 64.3 | 6893 | 4283 | 62.1 | kept |
| DEG_sRGES | edaravone | 12440 | 1918 | 15.4 | 6893 | 2249 | 32.6 | kept |
| DEG_sRGES | dextromethorphan | 12440 | 8246 | 66.3 | 6893 | 5880 | 85.3 | kept |
| DEG_sRGES | quinidine | 12440 | 7806 | 62.7 | 6893 | 4797 | 69.6 | kept |
| DEG_sRGES | phenylbutyrate | 12440 | 6614 | 53.2 | 6893 |  |  | removed |
| DEG_wsRGES | riluzole | 12419 | 8568 | 69.0 | 6731 | 4835 | 71.8 | kept |
| DEG_wsRGES | edaravone | 12419 | 2490 | 20.0 | 6731 | 2285 | 33.9 | kept |
| DEG_wsRGES | dextromethorphan | 12419 | 9298 | 74.9 | 6731 | 4707 | 69.9 | kept |
| DEG_wsRGES | quinidine | 12419 | 7706 | 62.1 | 6731 | 5392 | 80.1 | kept |
| DEG_wsRGES | phenylbutyrate | 12419 | 3206 | 25.8 | 6731 |  |  | removed |
| DEG_best | riluzole | 12440 | 587 | 4.7 | 6893 | 587 | 8.5 | kept |
| DEG_best | edaravone | 12440 | 3961 | 31.8 | 6893 | 3954 | 57.4 | kept |
| DEG_best | dextromethorphan | 12440 | 6716 | 54.0 | 6893 | 5960 | 86.5 | kept |
| DEG_best | quinidine | 12440 | 3787 | 30.4 | 6893 | 3782 | 54.9 | kept |
| DEG_best | phenylbutyrate | 12440 | 4530 | 36.4 | 6893 |  |  | removed |
| DEG_sRGES_topQ3 | riluzole | 676 | 386 | 57.1 | 37 |  |  | removed |
| DEG_sRGES_topQ3 | phenylbutyrate | 676 | 73 | 10.8 | 37 |  |  | removed |
| DEG_wsRGES_topQ3 | riluzole | 676 | 386 | 57.1 | 37 |  |  | removed |
| DEG_wsRGES_topQ3 | phenylbutyrate | 676 | 73 | 10.8 | 37 |  |  | removed |
| DEG_sRGES_topMed | riluzole | 1642 | 1271 | 77.4 | 489 | 415 | 84.9 | kept |
| DEG_sRGES_topMed | dextromethorphan | 1642 | 1294 | 78.8 | 489 |  |  | removed |
| DEG_sRGES_topMed | quinidine | 1642 | 1579 | 96.2 | 489 | 376 | 76.9 | kept |
| DEG_sRGES_topMed | phenylbutyrate | 1642 | 780 | 47.5 | 489 |  |  | removed |
| DEG_wsRGES_topMed | riluzole | 1642 | 1248 | 76.0 | 489 | 401 | 82.0 | kept |
| DEG_wsRGES_topMed | dextromethorphan | 1642 | 1260 | 76.7 | 489 |  |  | removed |
| DEG_wsRGES_topMed | quinidine | 1642 | 1563 | 95.2 | 489 | 374 | 76.5 | kept |
| DEG_wsRGES_topMed | phenylbutyrate | 1642 | 720 | 43.8 | 489 |  |  | removed |
| DEG_LM_TSR | riluzole | 3221 | 489 | 15.2 | 601 | 466 | 77.5 | kept |
| DEG_LM_TSR | dextromethorphan | 3221 | 1858 | 57.7 | 601 |  |  | removed |
| DEG_LM_TSR | quinidine | 3221 | 2214 | 68.7 | 601 |  |  | removed |
| DEG_LM_TSR | phenylbutyrate | 3221 | 3023 | 93.9 | 601 |  |  | removed |
| mith_TSR | riluzole | 3221 | 442 | 13.7 | 3219 | 442 | 13.7 | kept |
| mith_TSR | dextromethorphan | 3221 | 2326 | 72.2 | 3219 | 2326 | 72.3 | kept |
| mith_TSR | quinidine | 3221 | 1962 | 60.9 | 3219 | 1962 | 61.0 | kept |
| mith_TSR | phenylbutyrate | 3221 | 2755 | 85.5 | 3219 | 2753 | 85.5 | kept |
| mith_sRGES | riluzole | 12440 | 3306 | 26.6 | 12434 | 3309 | 26.6 | kept |
| mith_sRGES | edaravone | 12440 | 4562 | 36.7 | 12434 | 4569 | 36.7 | kept |
| mith_sRGES | dextromethorphan | 12440 | 8978 | 72.2 | 12434 | 8968 | 72.1 | kept |
| mith_sRGES | quinidine | 12440 | 8912 | 71.6 | 12434 | 8904 | 71.6 | kept |
| mith_sRGES | phenylbutyrate | 12440 | 9961 | 80.1 | 12434 | 9954 | 80.1 | kept |
| mith_wsRGES | riluzole | 12419 | 575 | 4.6 | 12413 | 576 | 4.6 | kept |
| mith_wsRGES | edaravone | 12419 | 4432 | 35.7 | 12413 | 4435 | 35.7 | kept |
| mith_wsRGES | dextromethorphan | 12419 | 10037 | 80.8 | 12413 | 10027 | 80.8 | kept |
| mith_wsRGES | quinidine | 12419 | 9465 | 76.2 | 12413 | 9452 | 76.1 | kept |
| mith_wsRGES | phenylbutyrate | 12419 | 11741 | 94.5 | 12413 | 11735 | 94.5 | kept |
| mith_best | riluzole | 12440 | 416 | 3.3 | 12434 | 416 | 3.3 | kept |
| mith_best | edaravone | 12440 | 690 | 5.5 | 12434 | 690 | 5.5 | kept |
| mith_best | dextromethorphan | 12440 | 3839 | 30.9 | 12434 | 3839 | 30.9 | kept |
| mith_best | quinidine | 12440 | 3969 | 31.9 | 12434 | 3969 | 31.9 | kept |
| mith_best | phenylbutyrate | 12440 | 3058 | 24.6 | 12434 | 3058 | 24.6 | kept |
| mith_sRGES_topQ3 | riluzole | 676 | 39 | 5.8 | 676 | 38 | 5.6 | kept |
| mith_sRGES_topQ3 | phenylbutyrate | 676 | 583 | 86.2 | 676 | 583 | 86.2 | kept |
| mith_wsRGES_topQ3 | riluzole | 676 | 39 | 5.8 | 676 | 38 | 5.6 | kept |
| mith_wsRGES_topQ3 | phenylbutyrate | 676 | 583 | 86.2 | 676 | 583 | 86.2 | kept |
| mith_sRGES_topMed | riluzole | 1642 | 145 | 8.8 | 1642 | 146 | 8.9 | kept |
| mith_sRGES_topMed | dextromethorphan | 1642 | 1259 | 76.7 | 1642 | 1259 | 76.7 | kept |
| mith_sRGES_topMed | quinidine | 1642 | 1573 | 95.8 | 1642 | 1573 | 95.8 | kept |
| mith_sRGES_topMed | phenylbutyrate | 1642 | 1474 | 89.8 | 1642 | 1474 | 89.8 | kept |
| mith_wsRGES_topMed | riluzole | 1642 | 84 | 5.1 | 1642 | 85 | 5.2 | kept |
| mith_wsRGES_topMed | dextromethorphan | 1642 | 1223 | 74.5 | 1642 | 1223 | 74.5 | kept |
| mith_wsRGES_topMed | quinidine | 1642 | 1543 | 94.0 | 1642 | 1543 | 94.0 | kept |
| mith_wsRGES_topMed | phenylbutyrate | 1642 | 1519 | 92.5 | 1642 | 1519 | 92.5 | kept |
| mith_LM_TSR | riluzole | 3221 | 368 | 11.4 | 862 | 368 | 42.7 | kept |
| mith_LM_TSR | dextromethorphan | 3221 | 1926 | 59.8 | 862 |  |  | removed |
| mith_LM_TSR | quinidine | 3221 | 1408 | 43.7 | 862 |  |  | removed |
| mith_LM_TSR | phenylbutyrate | 3221 | 3036 | 94.3 | 862 |  |  | removed |
| mith_LM_sRGES | riluzole | 12440 | 3699 | 29.7 | 5464 | 2511 | 46.0 | kept |
| mith_LM_sRGES | edaravone | 12440 | 2870 | 23.1 | 5464 | 1054 | 19.3 | kept |
| mith_LM_sRGES | dextromethorphan | 12440 | 8947 | 71.9 | 5464 | 5161 | 94.5 | kept |
| mith_LM_sRGES | quinidine | 12440 | 8026 | 64.5 | 5464 | 3412 | 62.4 | kept |
| mith_LM_sRGES | phenylbutyrate | 12440 | 8862 | 71.2 | 5464 | 2838 | 51.9 | kept |
| mith_LM_wsRGES | riluzole | 12419 | 1456 | 11.7 | 5255 | 1433 | 27.3 | kept |
| mith_LM_wsRGES | edaravone | 12419 | 3176 | 25.6 | 5255 | 1445 | 27.5 | kept |
| mith_LM_wsRGES | dextromethorphan | 12419 | 9728 | 78.3 | 5255 | 3683 | 70.1 | kept |
| mith_LM_wsRGES | quinidine | 12419 | 8048 | 64.8 | 5255 | 3722 | 70.8 | kept |
| mith_LM_wsRGES | phenylbutyrate | 12419 | 9365 | 75.4 | 5255 | 3049 | 58.0 | kept |
| mith_LM_best | riluzole | 12440 | 1502 | 12.1 | 5464 | 1502 | 27.5 | kept |
| mith_LM_best | edaravone | 12440 | 1848 | 14.9 | 5464 | 1847 | 33.8 | kept |
| mith_LM_best | dextromethorphan | 12440 | 5620 | 45.2 | 5464 | 5124 | 93.8 | kept |
| mith_LM_best | quinidine | 12440 | 2362 | 19.0 | 5464 | 2352 | 43.0 | kept |
| mith_LM_best | phenylbutyrate | 12440 | 2984 | 24.0 | 5464 | 2943 | 53.9 | kept |
| mith_LM_sRGES_topQ3 | riluzole | 676 | 106 | 15.7 | 40 |  |  | removed |
| mith_LM_sRGES_topQ3 | phenylbutyrate | 676 | 388 | 57.4 | 40 |  |  | removed |
| mith_LM_wsRGES_topQ3 | riluzole | 676 | 106 | 15.7 | 40 |  |  | removed |
| mith_LM_wsRGES_topQ3 | phenylbutyrate | 676 | 388 | 57.4 | 40 |  |  | removed |
| mith_LM_sRGES_topMed | riluzole | 1642 | 807 | 49.1 | 501 |  |  | removed |
| mith_LM_sRGES_topMed | dextromethorphan | 1642 | 1345 | 81.9 | 501 |  |  | removed |
| mith_LM_sRGES_topMed | quinidine | 1642 | 1594 | 97.1 | 501 | 439 | 87.6 | kept |
| mith_LM_sRGES_topMed | phenylbutyrate | 1642 | 1134 | 69.1 | 501 | 365 | 72.9 | kept |
| mith_LM_wsRGES_topMed | riluzole | 1642 | 728 | 44.3 | 501 |  |  | removed |
| mith_LM_wsRGES_topMed | dextromethorphan | 1642 | 1314 | 80.0 | 501 |  |  | removed |
| mith_LM_wsRGES_topMed | quinidine | 1642 | 1570 | 95.6 | 501 | 432 | 86.2 | kept |
| mith_LM_wsRGES_topMed | phenylbutyrate | 1642 | 1126 | 68.6 | 501 | 365 | 72.9 | kept |
| mith_pw_TSR | riluzole | 3221 | 411 | 12.8 | 2980 | 411 | 13.8 | kept |
| mith_pw_TSR | dextromethorphan | 3221 | 1305 | 40.5 | 2980 | 1306 | 43.8 | kept |
| mith_pw_TSR | quinidine | 3221 | 1470 | 45.6 | 2980 | 1471 | 49.4 | kept |
| mith_pw_TSR | phenylbutyrate | 3221 | 3124 | 97.0 | 2980 | 2883 | 96.7 | kept |
| mith_pw_sRGES | riluzole | 12440 | 3405 | 27.4 | 12221 | 3529 | 28.9 | kept |
| mith_pw_sRGES | edaravone | 12440 | 6423 | 51.6 | 12221 | 6380 | 52.2 | kept |
| mith_pw_sRGES | dextromethorphan | 12440 | 7861 | 63.2 | 12221 | 7938 | 65.0 | kept |
| mith_pw_sRGES | quinidine | 12440 | 8606 | 69.2 | 12221 | 8282 | 67.8 | kept |
| mith_pw_sRGES | phenylbutyrate | 12440 | 8468 | 68.1 | 12221 | 8259 | 67.6 | kept |
| mith_pw_wsRGES | riluzole | 12419 | 646 | 5.2 | 12198 | 686 | 5.6 | kept |
| mith_pw_wsRGES | edaravone | 12419 | 6271 | 50.5 | 12198 | 6237 | 51.1 | kept |
| mith_pw_wsRGES | dextromethorphan | 12419 | 7980 | 64.3 | 12198 | 8358 | 68.5 | kept |
| mith_pw_wsRGES | quinidine | 12419 | 8656 | 69.7 | 12198 | 8166 | 66.9 | kept |
| mith_pw_wsRGES | phenylbutyrate | 12419 | 11714 | 94.3 | 12198 | 11500 | 94.3 | kept |
| mith_pw_best | riluzole | 12440 | 490 | 3.9 | 12221 | 490 | 4.0 | kept |
| mith_pw_best | edaravone | 12440 | 6206 | 49.9 | 12221 | 6206 | 50.8 | kept |
| mith_pw_best | dextromethorphan | 12440 | 4511 | 36.3 | 12221 | 4511 | 36.9 | kept |
| mith_pw_best | quinidine | 12440 | 2818 | 22.7 | 12221 | 2818 | 23.1 | kept |
| mith_pw_best | phenylbutyrate | 12440 | 3177 | 25.5 | 12221 | 3177 | 26.0 | kept |
| mith_pw_sRGES_topQ3 | riluzole | 676 | 22 | 3.3 | 648 | 24 | 3.7 | kept |
| mith_pw_sRGES_topQ3 | phenylbutyrate | 676 | 506 | 74.9 | 648 | 481 | 74.2 | kept |
| mith_pw_wsRGES_topQ3 | riluzole | 676 | 22 | 3.3 | 648 | 24 | 3.7 | kept |
| mith_pw_wsRGES_topQ3 | phenylbutyrate | 676 | 506 | 74.9 | 648 | 481 | 74.2 | kept |
| mith_pw_sRGES_topMed | riluzole | 1642 | 62 | 3.8 | 1569 | 68 | 4.3 | kept |
| mith_pw_sRGES_topMed | dextromethorphan | 1642 | 819 | 49.9 | 1569 |  |  | removed |
| mith_pw_sRGES_topMed | quinidine | 1642 | 1133 | 69.0 | 1569 |  |  | removed |
| mith_pw_sRGES_topMed | phenylbutyrate | 1642 | 1425 | 86.8 | 1569 | 1359 | 86.6 | kept |
| mith_pw_wsRGES_topMed | riluzole | 1642 | 43 | 2.6 | 1569 | 49 | 3.1 | kept |
| mith_pw_wsRGES_topMed | dextromethorphan | 1642 | 832 | 50.7 | 1569 |  |  | removed |
| mith_pw_wsRGES_topMed | quinidine | 1642 | 1117 | 68.0 | 1569 |  |  | removed |
| mith_pw_wsRGES_topMed | phenylbutyrate | 1642 | 1467 | 89.3 | 1569 | 1394 | 88.8 | kept |

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
classes = sorted(k['class'].unique())
classes = sorted(k['class'].unique())
cmap = dict(zip(classes, ['#0000FF', '#FF0000', '#008000', '#FF8C00', '#8B00C8', '0.3']))
for ax, (title, sel) in zip(axes, [('TSR rankings', k['ranking'].isin(tsr_names)), ('per-instance rankings', ~k['ranking'].isin(tsr_names))]):
    d = k[sel & (k['status'] == 'kept')]
    for c in classes:
        e = d[d['class'] == c]
        ax.scatter(e['percentile_before'], e['percentile_after'], s=22, alpha=.7, color=cmap[c], label=f'{c} (n={e["drug"].nunique()} drugs)')
    ax.plot([0, 100], [0, 100], color='k', lw=.6); ax.set_title(title, fontsize=10)
    ax.set_xlabel('percentile, unfiltered ranking'); ax.set_xlim(0, 100); ax.set_ylim(0, 100)
    nrem = int(((k['status'] == 'removed') & sel).sum()); ax.text(.02, .96, f'{nrem} (drug, ranking) pairs removed', transform=ax.transAxes, fontsize=8, va='top')
axes[0].set_ylabel('percentile, BH-filtered ranking'); axes[0].legend(fontsize=7, loc='lower right')
plt.tight_layout()
f = IMG / f'{SLUG}_cs_pvalue_filter_known_drugs.png'
fig.savefig(f, dpi=130); plt.close(fig)
display(Markdown(f"![known drugs]({REL}/{f.name})\n\n**Figure 4.** Known drugs kept by the BH filter: percentile (0-100, low = stronger predicted reversal) in the unfiltered "
                 "ranking against the filtered one, one point per (drug, ranking); left TSR rankings, right per-instance rankings; the diagonal marks no change; removed pairs are counted in each panel."))

![known drugs](../imgs/case_studies/als_NYGC/als_NYGC_cs_pvalue_filter_known_drugs.png)

**Figure 4.** Known drugs kept by the BH filter: percentile (0-100, low = stronger predicted reversal) in the unfiltered ranking against the filtered one, one point per (drug, ranking); left TSR rankings, right per-instance rankings; the diagonal marks no change; removed pairs are counted in each panel.

## 5. Recomputation check

In [12]:
# (a) per-drug scores of the unfiltered recomputation vs the stored step-'srges' table
st = pd.read_csv(OUT / f'{SLUG}_srges_per_drug.tsv', sep='\t')
new = res['tables']['none']
j = new.merge(st, on=['config', 'collapse', 'pert_iname'], how='outer', suffixes=('_new', '_old'), indicator=True)
rows = []
for c in ['score', 'n_instances', 'n_cell_lines', 'n_pert_ids']:
    both = j['_merge'] == 'both'
    x, y = j.loc[both, f'{c}_new'].astype(float), j.loc[both, f'{c}_old'].astype(float)
    rows.append({'check': f'{SLUG}_srges_per_drug.tsv, column {c}', 'n_compared': int(x.notna().sum()),
                 'n_unmatched': int((j['_merge'] != 'both').sum()), 'max_abs_diff': float((x - y).abs().max()),
                 'pass': bool((x - y).abs().max() <= TOL and (j['_merge'] == 'both').all())})
chk = [pd.DataFrame(rows)]
# (b) long ranking table vs the stored drug rankings
fn = OUT / f'{SLUG}_drug_rankings.tsv'
if fn.exists():
    stored = pd.read_csv(fn, sep='\t', low_memory=False)
    r, oa, ob = pf.sanity(res['long']['none'], stored, TOL)
    chk.append(r.assign(check=lambda d: f'{fn.name}, column ' + d['column'], n_unmatched=oa + ob)
                 [['check', 'n_compared', 'n_unmatched', 'max_abs_diff', 'n_diff_above_tol', 'n_diff_not_near_tie', 'pass']])
    tsr_ids = stored[stored['ranking'].str.endswith('_TSR')].groupby('ranking')['cs_run_id'].first()
    ln = res['long']['none']
    now = ln[ln['ranking'].str.endswith('_TSR')].groupby('ranking')['cs_run_id'].first()
    chk.append(pd.DataFrame([{'check': f'{fn.name}, TSR run IDs', 'n_compared': len(tsr_ids), 'n_unmatched': int((tsr_ids != now.reindex(tsr_ids.index)).sum()),
                              'max_abs_diff': np.nan, 'pass': bool((tsr_ids == now.reindex(tsr_ids.index)).all())}]))

chk.append(pd.DataFrame([{'check': f"frozen offsets on all instances vs unfiltered scores, {r_.config}", 'n_compared': r_.n_compared, 'n_unmatched': 0, 'max_abs_diff': r_.max_abs_diff, 'pass': bool(r_.max_abs_diff <= TOL)} for r_ in res['frozen_check'].itertuples()]))
chk = pd.concat(chk, ignore_index=True)
chk.to_csv(OUT / f'{SLUG}_cs_pvalue_filter_recomputation_check.tsv', sep='\t', index=False)
display(Markdown(f"**Table 7.** Recomputation check: the unfiltered rankings rebuilt by this notebook against the tables written by the case-study pipeline "
                 f"(`{SLUG}_cs_pvalue_filter_recomputation_check.tsv`). `n_compared`: values compared (count); `n_unmatched`: rows present in only one table; `max_abs_diff`: largest absolute difference "
                 f"(score in CS units; ranks in positions; percentiles in percentile points); `n_diff_above_tol`: differences above {TOL:g}; `n_diff_not_near_tie`: those not explained by a near-tie "
                 f"(two compounds whose scores differ by at most {TOL:g}, so that their order depends on floating-point rounding); `pass`: all differences within tolerance or on near-ties.\n\n" + pf.md_table(chk, 3)))
display(Markdown(f"**All checks pass: {bool(chk['pass'].all())}.**"))

**Table 7.** Recomputation check: the unfiltered rankings rebuilt by this notebook against the tables written by the case-study pipeline (`als_NYGC_cs_pvalue_filter_recomputation_check.tsv`). `n_compared`: values compared (count); `n_unmatched`: rows present in only one table; `max_abs_diff`: largest absolute difference (score in CS units; ranks in positions; percentiles in percentile points); `n_diff_above_tol`: differences above 1e-12; `n_diff_not_near_tie`: those not explained by a near-tie (two compounds whose scores differ by at most 1e-12, so that their order depends on floating-point rounding); `pass`: all differences within tolerance or on near-ties.

| check | n_compared | n_unmatched | max_abs_diff | pass | n_diff_above_tol | n_diff_not_near_tie |
|---|---|---|---|---|---|---|
| als_NYGC_srges_per_drug.tsv, column score | 167772 | 0 | 1.00e-16 | True |  |  |
| als_NYGC_srges_per_drug.tsv, column n_instances | 167772 | 0 | 0.000 | True |  |  |
| als_NYGC_srges_per_drug.tsv, column n_cell_lines | 167772 | 0 | 0.000 | True |  |  |
| als_NYGC_srges_per_drug.tsv, column n_pert_ids | 167772 | 0 | 0.000 | True |  |  |
| als_NYGC_drug_rankings.tsv, column score | 183845 | 0 | 1.39e-16 | True | 0 | 0 |
| als_NYGC_drug_rankings.tsv, column rank | 183845 | 0 | 1.000 | True | 14 | 0 |
| als_NYGC_drug_rankings.tsv, column percentile | 183845 | 0 | 0.074 | True | 28 | 0 |
| als_NYGC_drug_rankings.tsv, column percentile_shared_DEG | 63776 | 0 | 0.047 | True | 6 | 0 |
| als_NYGC_drug_rankings.tsv, column n_instances | 167740 | 0 | 0.000 | True | 0 | 0 |
| als_NYGC_drug_rankings.tsv, column n_cell_lines | 167740 | 0 | 0.000 | True | 0 | 0 |
| als_NYGC_drug_rankings.tsv, column n_drugs_stratum | 167740 | 0 | 0.000 | True | 0 | 0 |
| als_NYGC_drug_rankings.tsv, column percentile_stratum | 167740 | 0 | 0.439 | True | 16 | 0 |
| als_NYGC_drug_rankings.tsv, TSR run IDs | 5 | 0 |  | True |  |  |
| frozen offsets on all instances vs unfiltered scores, DEG | 41943 | 0 | 0.000 | True |  |  |
| frozen offsets on all instances vs unfiltered scores, mith | 41943 | 0 | 0.000 | True |  |  |
| frozen offsets on all instances vs unfiltered scores, mith_LM | 41943 | 0 | 0.000 | True |  |  |
| frozen offsets on all instances vs unfiltered scores, mith_pw | 41943 | 0 | 0.000 | True |  |  |

**All checks pass: True.**

## 6. Outputs

In [13]:
tidy = res['long']['bh05']
tidy.to_csv(OUT / f'{SLUG}_drug_rankings_pvalue_filtered.tsv', sep='\t', index=False)
if chg['abs_diff_spearman_variants'].max() > 0.05:
    res['long']['bh05f'].to_csv(OUT / f'{SLUG}_drug_rankings_pvalue_filtered_frozen_offsets.tsv', sep='\t', index=False)
files = sorted(OUT.glob(f'{SLUG}_cs_pvalue_filter_*')) + [OUT / f'{SLUG}_drug_rankings_pvalue_filtered.tsv'] + sorted(OUT.glob(f'{SLUG}_drug_rankings_pvalue_filtered_frozen*')) + sorted(IMG.glob(f'{SLUG}_cs_pvalue_filter_*'))
display(Markdown(f"Filtered rankings (BH-adjusted p <= 0.05) of every ranking, same columns as `{SLUG}_drug_rankings.tsv` "
                 f"(`ranking`, `config`, `collapse`, `drug_key`, `drug`, `score`, `rank`, `percentile`, `percentile_shared_DEG`, `n_instances`, `n_cell_lines`, `stratum`, `n_drugs_stratum`, `percentile_stratum`, `cs_run_id`): "
                 f"{len(tidy):,} rows, {tidy['ranking'].nunique()} rankings; rank and percentiles are computed within the filtered universe, `percentile_shared_DEG` over the compounds scored by both filtered `DEG_TSR` and `DEG_sRGES`, "
                 f"`n_instances` counts the surviving instances.\n\n**Files:**\n\n" + "\n".join(f"- `{str(p.relative_to(ROOT))}`" for p in files)))

Filtered rankings (BH-adjusted p <= 0.05) of every ranking, same columns as `als_NYGC_drug_rankings.tsv` (`ranking`, `config`, `collapse`, `drug_key`, `drug`, `score`, `rank`, `percentile`, `percentile_shared_DEG`, `n_instances`, `n_cell_lines`, `stratum`, `n_drugs_stratum`, `percentile_stratum`, `cs_run_id`): 132,249 rows, 33 rankings; rank and percentiles are computed within the filtered universe, `percentile_shared_DEG` over the compounds scored by both filtered `DEG_TSR` and `DEG_sRGES`, `n_instances` counts the surviving instances.

**Files:**

- `other_outputs/case_studies/als_NYGC/als_NYGC_cs_pvalue_filter_cell_lines.tsv`
- `other_outputs/case_studies/als_NYGC/als_NYGC_cs_pvalue_filter_instances_kept.tsv`
- `other_outputs/case_studies/als_NYGC/als_NYGC_cs_pvalue_filter_known_drugs.tsv`
- `other_outputs/case_studies/als_NYGC/als_NYGC_cs_pvalue_filter_offset_variants.tsv`
- `other_outputs/case_studies/als_NYGC/als_NYGC_cs_pvalue_filter_recomputation_check.tsv`
- `other_outputs/case_studies/als_NYGC/als_NYGC_cs_pvalue_filter_resolution.tsv`
- `other_outputs/case_studies/als_NYGC/als_NYGC_cs_pvalue_filter_runs.tsv`
- `other_outputs/case_studies/als_NYGC/als_NYGC_cs_pvalue_filter_sign_split.tsv`
- `other_outputs/case_studies/als_NYGC/als_NYGC_cs_pvalue_filter_summary.tsv`
- `other_outputs/case_studies/als_NYGC/als_NYGC_drug_rankings_pvalue_filtered.tsv`
- `imgs/case_studies/als_NYGC/als_NYGC_cs_pvalue_filter_agreement.png`
- `imgs/case_studies/als_NYGC/als_NYGC_cs_pvalue_filter_fraction_kept.png`
- `imgs/case_studies/als_NYGC/als_NYGC_cs_pvalue_filter_known_drugs.png`
- `imgs/case_studies/als_NYGC/als_NYGC_cs_pvalue_filter_pvalue_distribution.png`